In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pass del transpiler AI
I pass del transpiler potenziati dall'AI sono pass che funzionano come sostituti diretti dei pass "tradizionali" di Qiskit per alcune operazioni di transpilazione. Spesso producono risultati migliori rispetto agli algoritmi euristici esistenti (ad esempio, profondità e conteggio CNOT inferiori), ma sono anche molto più veloci rispetto ad algoritmi di ottimizzazione come i risolutori di soddisfacibilità booleana. I pass del transpiler AI possono essere eseguiti nel tuo ambiente locale o nel cloud tramite il Qiskit Transpiler Service, se fai parte del Piano Premium, del Piano Flex o del Piano On-Prem (tramite IBM Quantum Platform API) di IBM Quantum&reg;.

> **Note:** I pass del transpiler potenziati dall'AI sono in stato di rilascio beta e soggetti a modifiche.
>     Se hai feedback o vuoi contattare il team di sviluppo, utilizza questo [canale dello Spazio Slack di Qiskit](https://qiskit.slack.com/archives/C06KF8YHUAU).

I seguenti pass sono attualmente disponibili:

**Pass di routing**
 - `AIRouting`: Selezione del layout e routing del circuito

**Pass di sintesi del circuito**
 - `AICliffordSynthesis`: Sintesi di circuiti Clifford
 - `AILinearFunctionSynthesis`: Sintesi di circuiti a funzione lineare
 - `AIPermutationSynthesis`: Sintesi di circuiti di permutazione
 - `AIPauliNetworkSynthesis`: Sintesi di circuiti Pauli Network

Per usare i pass del transpiler AI, prima [installa il pacchetto `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Visita la [documentazione API di qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) per maggiori informazioni sulle diverse opzioni disponibili.

## Eseguire i pass del transpiler AI in locale o nel cloud
Se vuoi utilizzare i pass del transpiler potenziati dall'AI nel tuo ambiente locale gratuitamente, installa `qiskit-ibm-transpiler` con alcune dipendenze aggiuntive come segue:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Senza queste dipendenze aggiuntive, i pass del transpiler potenziati dall'AI vengono eseguiti nel cloud tramite il Qiskit Transpiler Service (disponibile solo per gli utenti del Piano Premium, del Piano Flex o del Piano On-Prem (tramite IBM Quantum Platform API) di IBM Quantum). Dopo aver installato le dipendenze aggiuntive, la modalità predefinita per eseguire i pass del transpiler potenziati dall'AI è utilizzare il tuo computer locale.

## Pass di routing AI
Il pass `AIRouting` agisce sia come fase di layout che come fase di routing. Può essere utilizzato all'interno di un `PassManager` come segue:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

In questo esempio, il `backend` determina per quale coupling map effettuare il routing, il parametro `optimization_level` (1, 2 o 3) determina lo sforzo computazionale da impiegare nel processo (valori più alti di solito producono risultati migliori ma richiedono più tempo), e il parametro `layout_mode` specifica come gestire la selezione del layout.
Il parametro `layout_mode` include le seguenti opzioni:

- `keep`: Questa modalità rispetta il layout impostato dai pass del transpiler precedenti (o usa il layout triviale se non impostato). Viene tipicamente usata solo quando il circuito deve essere eseguito su qubit specifici del dispositivo. Spesso produce risultati peggiori perché ha meno margine per l'ottimizzazione.
- `improve`: Questa modalità utilizza il layout impostato dai pass del transpiler precedenti come punto di partenza. È utile quando hai una buona ipotesi iniziale per il layout; ad esempio, per circuiti costruiti in modo da seguire approssimativamente la coupling map del dispositivo. È utile anche se vuoi provare altri pass di layout specifici in combinazione con il pass `AIRouting`.
- `optimize`: Questa è la modalità predefinita. Funziona meglio per circuiti generali in cui potresti non avere buone ipotesi di layout. Questa modalità ignora le selezioni di layout precedenti.
- `local_mode`: Questo flag determina dove viene eseguito il pass `AIRouting`. Se `False`, `AIRouting` viene eseguito in remoto tramite il Qiskit Transpiler Service. Se `True`, il pacchetto tenta di eseguire il pass nel tuo ambiente locale, con un fallback alla modalità cloud se le dipendenze richieste non vengono trovate.

## Pass di sintesi del circuito AI
I pass di sintesi del circuito AI ti consentono di ottimizzare porzioni di diversi tipi di circuito ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) ri-sintetizzandole. Un modo tipico per utilizzare il pass di sintesi è il seguente:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

La sintesi rispetta la coupling map del dispositivo: può essere eseguita in sicurezza dopo altri pass di routing senza alterare il circuito, quindi il circuito complessivo rispetterà comunque i vincoli del dispositivo. Per impostazione predefinita, la sintesi sostituirà il sotto-circuito originale solo se il sotto-circuito sintetizzato migliora l'originale (attualmente viene verificato solo il conteggio CNOT), ma è possibile forzare la sostituzione sempre impostando `replace_only_if_better=False`.

I seguenti pass di sintesi sono disponibili da `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Sintesi per circuiti [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blocchi di gate `H`, `S` e `CX`). Attualmente fino a blocchi di nove qubit.
- *AILinearFunctionSynthesis*: Sintesi per circuiti [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blocchi di gate `CX` e `SWAP`). Attualmente fino a blocchi di nove qubit.
- *AIPermutationSynthesis*: Sintesi per circuiti [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blocchi di gate `SWAP`). Attualmente disponibile per blocchi di 65, 33 e 27 qubit.
- *AIPauliNetworkSynthesis*: Sintesi per circuiti Pauli Network (blocchi di gate `H`, `S`, `SX`, `CX`, `RX`, `RY` e `RZ`). Attualmente fino a blocchi di sei qubit.

Prevediamo di aumentare gradualmente la dimensione dei blocchi supportati.

Tutti i pass utilizzano un pool di thread per inviare più richieste in parallelo. Per impostazione predefinita, il numero massimo di thread corrisponde al numero di core più quattro (valori predefiniti per l'oggetto Python `ThreadPoolExecutor`). Puoi tuttavia impostare il tuo valore tramite l'argomento `max_threads` all'istanziazione del pass. Ad esempio, la seguente riga istanzia il pass `AILinearFunctionSynthesis` consentendogli di utilizzare un massimo di 20 thread.